# Dynamics of Railway Bridges - Implementation Notes

Python implementations of vibration analysis models for railway bridges.
Theoretical background and equations reference:

- Fryba, L. *Dynamics of Railway Bridges*. Thomas Telford.

Refer to the original text for derivations and theory.


<a id="preface"></a>
# Preface

The purpose of this notebook is to hold my notes while working through the contents of Ladislav Freyba's *Dynamics of Railway Bridges*. The Book focuses on modeling railway dynamics. The book specfically leaves out code examples and focuses on the mathematical principles and explaining them. This notebook, is an attempt to put actual code to those concepts, to later be transferred into the owner's [civiply](https://daneparks.com/Dane/civilpy) python package. Other languages may be used if necessary, but the intent of the author is to do as much of the following programming as possible utilizing 'pure' python. Mathemetical libraries and imports will be added as necessary and the libraries will be imported near the top of the file. It is up to the user to get a python environment configured in a way that enables the use of this file, that is beyond the scope of this document.

Main points - Book was published in 1996 based on author's experience with bridges in Central Europe (Czech and Slovak Republics) it focuses on traditional freight and passenger trains not exceeding 500 km/h. Since the book is largely in metric units, and the Author of this notebook/python library is American, the units will commonly be converted as follows,

In [1]:
import pint

units = pint.UnitRegistry()
speed = 500 * units('km/h')
speed.to('mph')

The author's environment utilizes the following python version,

In [2]:
import sys
print(f"Python Version: {sys.version_info[0]}.{sys.version_info[1]}")

Occasionally packages like matplotlib will output warnings for things like dividing by 0. You can suppress these warnings as follows,

In [3]:
import warnings
warnings.filterwarnings("ignore")

A few additional libraries will be utilized to aid in the calculations performed in this notebook,

In [4]:
import math
import matplotlib.pyplot as plt

<a id="notation"></a>
# Notation

The following is an outline of the notation utilized in the book as well as in the python package. Due to python variables having the requirement to be unique and not contain greek letters, the repeated values had to be expanded to more verbose python varibles for use in coding namespaces. In the spirit of python, the variables were made as 'pythonic' as possible in an attempt to sacrifice brevity for clarity. The following outline provides the books notation, the python variable, and then the books description of that variable. Formulas from the book will first be shown in their traditional mathematical representation,

$$ a^2 + b^2 = c^2 $$

followed by their python representation when solving the equation;

```python
c = np.sqrt(a**2 + b**2)
```

for more complex equations, expect numpy to be utilized for variable mathematics and sympy for calculus type mathematics.

In [5]:
import numpy as np
import sympy as sp

The next cell centers the outputs of figures in jupyter.

# Notes

<a id="intro"></a>
## Introduction

```mermaid
flowchart TD

A(["Dynamic Effects of Railway Vehicles on Bridges"]) --> B(["Deterministic"])
A --> C(["Stochastic also due to traffic flow"])
B --> D([" "])
C --> D([" "])
D --> E(["Vertical effects"])
D --> F(["Horizontal Effects"])
F --> G(["Transverse"])
F --> H(["Longitudinal"])
E --> I(["Influence of moving<br>Forces and masses"])
E --> J(["Irregularities"])
J --> K(["Of the tracks"])
J --> L(["Of vehicle wheels"])
H --> M(["Starting Forces"])
H --> N(["Braking Forces"])
G --> O(["Lateral Impacts"])
G --> P(["centrifugal Forces"])
```

In [6]:
def damped_vibration(m, c, k, x0, v0, t):
    """
    Models the natural damped vibration of a system with one degree of freedom.
    
    Parameters:
    - m : mass of the system
    - c : damping coefficient
    - k : stiffness of the system
    - x0: initial displacement
    - v0: initial velocity
    - t : time array
    
    Returns:
    - x : Displacement Array over Time t
    - omega_n : The natural frequency
    - zeta : Damping Ratio
    """
    # Calculate damped natural frequency
    omega_n = np.sqrt(k / m)
    zeta = c / (2 * np.sqrt(m * k))
    omega_d = omega_n * np.sqrt(1 - zeta ** 2)

    # Calculate constants A and B from initial conditions
    A = x0
    B = (v0 + zeta * omega_n * x0) / omega_d

    # Calculate the displacement x(t) over time
    x = np.exp(-zeta * omega_n * t) * (A * np.cos(omega_d * t) + B * np.sin(omega_d * t))

    return x, omega_n, zeta

In [7]:
# Example usage:
m = 1.0  # mass (kg)
c = 0.2  # damping coefficient (N*s/m)
k = 2.0  # stiffness (N/m)
x0 = 1.0  # initial displacement (m)
v0 = 0.0  # initial velocity (m/s)
t = np.linspace(0, 10, 1000)  # time array from 0 to 10 seconds

# Calculate displacement over time
x, omega_n, zeta = damped_vibration(m, c, k, x0, v0, t)

# Calculate the boundary lines
envelope_pos = np.exp(-zeta * omega_n * t)
envelope_neg = -np.exp(-zeta * omega_n * t)

In [8]:
# Plot the result
plt.plot(t, x, label='Damped Vibration')
plt.plot(t, envelope_pos, 'r--', label='Envelope +', alpha=0.7)
plt.plot(t, envelope_neg, 'r--', label='Envelope -', alpha=0.7)

plt.title('Fig. 1.3 Time-history of natural damped vibration of a system with one degree of freedom at sub-critical damping', y=-0.2)
plt.xlabel('Time (s)')
plt.ylabel('v(t)/v(0)')

# Add text at the center of the top boundary line
mid_time = t[len(t) // 2]
mid_envelope = np.exp(-zeta * omega_n * mid_time)
plt.text(mid_time+.5, mid_envelope, r'$e^{-\omega_b t}$', horizontalalignment='center', verticalalignment='bottom', fontsize=12)

plt.axhline(0, color='black', linestyle='-', linewidth=1)  # x-axis at y=0
plt.legend()
plt.grid(True)
plt.show()

### Dynamic Coefficient

In [9]:
def dynamic_coefficient(omega, omega_0, beta):
    """
    Calculate the dynamic coefficient (Delta) for a given excitation frequency
    and damping ratio.

    Parameters:
    - omega : array of excitation frequencies
    - omega_0: natural frequency of the system
    - beta : damping ratio

    Returns:
    - Delta : dynamic coefficient array
    """
    r = omega / omega_0
    Delta = 1 / np.sqrt((1 - r ** 2) ** 2 + (2 * beta * r) ** 2)
    return Delta

In [10]:
# Example usage:
omega_0 = 1.0  # normalized natural frequency
omegas = np.linspace(0, 3, 1000)  # normalized excitation frequencies (omega / omega_0)
beta_values = [0.0, 0.05, 0.1, 0.2, 0.3, 0.5, 1.0]  # different damping ratios

In [11]:
# Plot the resonance curves for different damping ratios
plt.figure(figsize=(10, 6))

for beta in beta_values:
    Delta = dynamic_coefficient(omegas, omega_0, beta)
    plt.plot(omegas, Delta, label=f'$\\beta$ = {beta}')

plt.title('Fig. 1.4 - Resonance Curves for Different Damping Ratios', y=-0.2)
plt.xlabel('$\\omega / \\omega_0$')
plt.ylabel('Dynamic Coefficient $\\delta$')
plt.legend()
plt.grid(True)
plt.ylim(0, 10)  # adjust y-axis limits for better visualization
plt.show()

### Stochastic Vibration

In [12]:
# Generate random processes x(t) and y(t)
np.random.seed(0)  # for reproducibility
t = np.linspace(0, 10, 1000)  # time array
x_t = np.sin(t) + np.random.normal(0, 0.5, len(t))  # random process x(t)
y_t = np.cos(t) + np.random.normal(0, 0.5, len(t))  # random process y(t)

# Define t1 and t2
t1 = 2.0  # time t1
t2 = 5.0  # time t2

# Calculate correlation
correlation = np.correlate(x_t, y_t, mode='full') / len(t)
lags = np.arange(-len(t) + 1, len(t))

# Plot the random processes
plt.figure(figsize=(12, 6))

plt.subplot(2, 1, 1)
plt.plot(t, x_t, label='x(t)')
plt.plot(t, y_t, label='y(t)')
plt.axvline(t1, color='r', linestyle='--', label='$t_1$')
plt.axvline(t2, color='g', linestyle='--', label='$t_2$')
plt.title('Random Processes x(t) and y(t)')
plt.xlabel('Time (s)')
plt.ylabel('Amplitude')
plt.legend()
plt.grid(True)

# Plot the correlation
plt.subplot(2, 1, 2)
plt.plot(lags, correlation)
plt.title('Correlation of x(t) and y(t)')
plt.xlabel('Fig. 1.5 Correlation of random processes x(t) and y(t) at times $t_1$ and $t_2$')
plt.ylabel('Correlation')
plt.grid(True)

plt.tight_layout()
plt.show()

<a id="bridge_models"></a>
## Theoretical Bridge Models

<a id="railway_vehicles"></a>
## Modelling of Railway Vehicles

<a id="frequencies"></a>
## Natural Frequencies of Railway Bridges

<a id="damping"></a>
## Damping of Railway Bridges

<a id="vehicle_speed"></a>
## Influence of Vehicle Speed on Dynamic Stresses of Bridges

<a id="track_irregularities"></a>
## Influence of Track Irregularities and Other Parameters

<a id="euler_method_notes"></a>

**Euler Method**

Euler's Method is one of the simplest and most straightforward numerical methods for solving ODEs. It is an initial value problem-solving method.

**Basic Idea**

Euler's method approximates the solution of the ODE by stepping forward in small, fixed intervals (step size (h)) from an initial starting point. It uses the slope at the current point to estimate the next point.

**Given an ODE:**

$ \frac{dy}{dt} = f(t, y(t)) $ with an initial condition $ y(t_0) = y_0 $, Euler's method updates the solution iteratively as follows:

**Algorithm**

1. Step Initialization: Choose a step size ( h ).

2. Iterative Update:

   $ y_{n+1} = y_n + h \cdot f(t_n, y_n)$

   $ t_{n+1} = t_n + h $

where $ y_n $ is the current approximation of the solution at time $ t_n $.

**Pros and Cons**

- **Pros:** Simple to implement, requires only the computation of the function ( f ).
- **Cons:** Low accuracy, requires very small step sizes to achieve reasonable accuracy, leading to high computational cost for long intervals.

<a id="runge_kutta_method_notes"></a>
**Runge-Kutta Method**

Runge-Kutta Methods are a family of higher-order, more accurate numerical methods for solving ODEs. The most commonly used is the 4th-order Runge-Kutta method (RK4), which provides a good balance between accuracy and computational efficiency.

Basic Idea

Runge-Kutta methods improve upon Euler's method by evaluating the slope at several points within the step interval and taking a weighted average of these slopes to estimate the next point.

Given an ODE:
$ \frac{dy}{dt} = f(t, y(t)) $ with an initial condition $ y(t_0) = y_0 $, the 4th-order Runge-Kutta method updates the solution as follows:

Algorithm (RK4)

1. Step Initialization: Choose a step size ( h ).
2. Compute Intermediate Slopes:
   $$ k_1 = f(t_n, y_n) $$
   $$ k_2 = f(t_n + \frac{h}{2}, y_n + \frac{h}{2} k_1) $$
   $$ k_3 = f(t_n + \frac{h}{2}, y_n + \frac{h}{2} k_2) $$
   $$ k_4 = f(t_n + h, y_n + h k_3) $$
4. Iterative Update:
   $$ y_{n+1} = y_n + \frac{h}{6} (k_1 + 2k_2 + 2k_3 + k_4) $$
   $$ t_{n+1} = t_n + h $$

**Pros and Cons**

- **Pros:** Higher accuracy than Euler's method, relatively easy to implement, and requires fewer steps for the same accuracy.
- **Cons:** More computational effort per step due to the evaluation of multiple slopes, but generally more efficient overall due to fewer steps needed.

**Example Comparison**

To illustrate the differences between Euler's method and the 4th-order Runge-Kutta method, let's solve a simple ODE:
$ \frac{dy}{dt} = t + y $

with initial condition $ y(0) = 1 $ over the interval $ 0 \leq t \leq 1 $.

Below is an example implementation of both methods in Python:

In [28]:
# Define the differential equation
def f(t, y):
    return t + y


# Exact solution for comparison
def exact_solution(t):
    return -t - 1 + 2 * np.exp(t)


# Euler's method
def euler_method(f, t0, y0, h, n):
    t = np.zeros(n + 1)
    y = np.zeros(n + 1)
    t[0], y[0] = t0, y0
    for i in range(n):
        t[i + 1] = t[i] + h
        y[i + 1] = y[i] + h * f(t[i], y[i])
    return t, y


# Runge-Kutta 4th order method (RK4)
def rk4_method(f, t0, y0, h, n):
    t = np.zeros(n + 1)
    y = np.zeros(n + 1)
    t[0], y[0] = t0, y0
    for i in range(n):
        k1 = f(t[i], y[i])
        k2 = f(t[i] + h / 2, y[i] + h * k1 / 2)
        k3 = f(t[i] + h / 2, y[i] + h * k2 / 2)
        k4 = f(t[i] + h, y[i] + h * k3)
        t[i + 1] = t[i] + h
        y[i + 1] = y[i] + h / 6 * (k1 + 2 * k2 + 2 * k3 + k4)
    return t, y


# Parameters
t0, y0 = 0, 1
h = 0.1
n = int(1 / h)

# Solve using Euler's method
t_euler, y_euler = euler_method(f, t0, y0, h, n)

# Solve using Runge-Kutta 4th order method
t_rk4, y_rk4 = rk4_method(f, t0, y0, h, n)

# Exact solution
t_exact = np.linspace(t0, 1, 100)
y_exact = exact_solution(t_exact)

# Plotting the results
plt.plot(t_exact, y_exact, 'k-', label='Exact Solution')
plt.plot(t_euler, y_euler, 'b.--', label="Euler's Method")
plt.plot(t_rk4, y_rk4, 'r.--', label='RK4 Method')
plt.xlabel('t')
plt.ylabel('y(t)')
plt.legend()
plt.title('Comparison of Euler and RK4 Methods')
plt.show()

In [55]:
# Calculate the static deformation v_0 at the midpoint of the beam
def calculate_v_0(F, l, E, I):
    return F * l ** 3 / (48 * EI)

# Calculate the static deformation v_0 at the midpoint of the beam
def calculate_u_0(F, l, E, A):
    return F * l / (E*A)

def calculate_omega_1(l, E, I, rho, A):
    return ((1.875**2 / (2 * np.pi * l **2)) * np.sqrt((E*I)/(rho*A)))

In [59]:
omega_1 = calculate_omega_1(
    l=1*units.meter, 
    E=20.5e10*units("N/m^2"), 
    I=1000*units('mm^4'), 
    rho=7.83e3*units('kg/m^3'), 
    A=120*units('mm^2')
)
omega_1.to('Hz')

<a id="motion_of_disc_parameters"></a>
#### Influence of Some Parameters

<a id="quasistatic_model"></a>
### Quasistatic Model

<a id="quasistatic_model_solution"></a>
#### Solution

<a id="quasistatic_model_parameters"></a>
#### Influence of Some Parameters

<a id="quasistatic_model_experiments"></a>
#### Experiments on Bridges

<a id="start_and_braking_forces"></a>
### Starting and Braking Forces on Bridges

<a id="transverse_effects"></a>
## Horizontal Transverse Effects on Bridges

<a id="transverse_beam_effects"></a>
### Beam

<a id="transverse_vertical_vibration"></a>
#### Vertical Vibration

<a id="transverse_Horizontal_vibration"></a>
#### Horizontal Vibration

<a id="transverse_torsional_vibration"></a>
#### Torsional Vibration

<a id="thin_walled_bar_vert_axis"></a>
### Thin-Walled Bar with Vertical Axis of Symmetry

<a id="thin_walled_bar_vert_axis_apprx_solution"></a>
#### Approximate Solution

<a id="thin_walled_bar_vert_axis_dynam_solution"></a>
#### Dynamic Solution

<a id="thin_walled_bar_two_axes"></a>
#### Thin-Walled Bar with Two Axes of Symmetry

<a id="horiz_trans_expiriments"></a>
### Experiments on Bridges

<a id="horiz_trans_coefficents"></a>
### Coefficients of Variation for Horizontal and Torsional Vibrations

<a id="horiz_trans_centrifugal"></a>
### Centrifugal Forces

<a id="traffic_loads"></a>
## Traffic Loads on Railway Bridges

<a id="counting_methods"></a>
## Statistical Counting methods for the Classification of Random Stress-Time History

<a id="stress_ranges"></a>
## Stress Ranges in Steel Railway Bridges

<a id="fatigue_assessment"></a>
## The Assessment of Steel Railway Bridges for Fatigue

<a id="appendix"></a>
## Appendix